# Kaggle GeoAI Fine-Tune: Da Nang Urban z18 Buildings from Overture Maps

This notebook fine-tunes OpenGeoAI/GeoAI Mask R-CNN building footprint detection with filtered weak labels from Overture Maps. It uses the z18 urban district rasters for Hai Chau, Thanh Khe, Son Tra, Ngu Hanh Son, Cam Le, and Lien Chieu, and intentionally excludes the Hoa Vang z16 raster.

The notebook searches `/kaggle/input/**` automatically and writes generated data, DDP training logs, models, smoke-test outputs, and the final zip artifact under `/kaggle/working/geoai_finetune/`.

## 1. Install dependencies

Kaggle GPU images already include CUDA-enabled PyTorch. This cell installs `geoai-py==0.37.2` and geospatial dependencies without requesting a PyTorch package explicitly.

In [ ]:
import subprocess
import sys

try:
    import torch
except ImportError as exc:
    raise RuntimeError('Kaggle GPU notebooks should provide CUDA-enabled PyTorch. Enable GPU before running.') from exc

print('[step 1/9] Install dependencies')
print(f'[setup] torch={torch.__version__}')
print(f'[setup] cuda_available={torch.cuda.is_available()} cuda_devices={torch.cuda.device_count()}')

PACKAGES = [
    'geoai-py==0.37.2',
    'geopandas>=0.14',
    'rasterio>=1.3',
    'shapely>=2.0',
    'pyogrio>=0.7',
    'fiona>=1.9',
    'rtree',
    'tqdm',
    'opencv-python-headless',
    'scikit-image',
    'matplotlib',
    'pillow',
    'timm',
    'torchgeo',
    'leafmap',
    'overturemaps',
]

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '--upgrade-strategy',
    'only-if-needed',
    *PACKAGES,
])

print('[setup] dependencies installed')

## 2. Config and imports

Defaults target a 4-5 GB z18 weak-label dataset on Kaggle T4 x2. `BATCH_SIZE_PER_GPU=2` gives global batch 4 under DDP.

In [ ]:
from pathlib import Path
from collections import defaultdict
import ast
import json
import math
import datetime
import os
import random
import shutil
import subprocess
import sys
import time
import unicodedata
import zipfile

import geoai
import geopandas as gpd
import numpy as np
import rasterio
import torch
from rasterio.features import rasterize
from rasterio.windows import Window

DISTRICT_IDS = ['haichau', 'thanhkhe', 'sontra', 'nguhanhson', 'camle', 'lienchieu']
EXCLUDED_RASTERS = ['hoavang_z16.tif']
SMOKE_DISTRICT_ID = 'haichau'

TILE_SIZE = 512
STRIDE = 384
MAX_TILES_PER_DISTRICT = 900
EPOCHS = 12
BATCH_SIZE_PER_GPU = 2
VAL_SPLIT = 0.15
TEST_SPLIT = 0.10
TEST_EVAL_MAX_TILES = 100

USE_DDP = True
MAX_DDP_GPUS = 2
NUM_WORKERS_PER_GPU = 2

LEARNING_RATE = 0.001
LR_STEP_SIZE = 4
LR_GAMMA = 0.5

SEED = 42
SMOKE_WINDOW_SIZE = 1024
CONFIDENCE_THRESHOLD = 0.6

LABEL_CONFIDENCE_MIN = 0.75
BUILDING_AREA_MIN_M2 = 20
BUILDING_AREA_MAX_M2 = 5000
NEGATIVE_BUFFER_M = 0.75

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'kaggle_working'
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
RUN_ROOT = WORK_ROOT / 'geoai_finetune'
DATASET_ROOT = RUN_ROOT / 'danang_urban_z18_dataset'
RAW_TILES_ROOT = DATASET_ROOT / '_district_exports'
VECTOR_DIR = DATASET_ROOT / 'vectors'
IMAGES_DIR = DATASET_ROOT / 'images'
LABELS_DIR = DATASET_ROOT / 'labels'
TRAIN_IMAGES_DIR = DATASET_ROOT / 'train_images'
TRAIN_LABELS_DIR = DATASET_ROOT / 'train_labels'
TEST_IMAGES_DIR = DATASET_ROOT / 'test_images'
TEST_LABELS_DIR = DATASET_ROOT / 'test_labels'
MODEL_DIR = RUN_ROOT / 'models' / 'danang_urban_z18_maskrcnn'
SMOKE_DIR = RUN_ROOT / 'smoke_test'
ZIP_PATH = WORK_ROOT / 'danang_urban_z18_geoai_maskrcnn.zip'
DDP_TRAIN_SCRIPT = RUN_ROOT / 'ddp_train_maskrcnn.py'

TRAIN_DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
CUDA_DEVICE_COUNT = torch.cuda.device_count()
DDP_WORLD_SIZE = min(MAX_DDP_GPUS, CUDA_DEVICE_COUNT) if USE_DDP and CUDA_DEVICE_COUNT >= 2 else 1
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_GPU * max(1, DDP_WORLD_SIZE)

PIPELINE_TOTAL_STEPS = 9


def log_step(step, title):
    print(f'\n[step {step}/{PIPELINE_TOTAL_STEPS}] {title}')


def log_detail(message):
    print(f'  - {message}')


log_step(2, 'Configure multi-district z18 DDP run')
print(f'[config] districts={DISTRICT_IDS}')
print(f'[config] excluded={EXCLUDED_RASTERS}')
print(f'[config] tile_size={TILE_SIZE} stride={STRIDE} max_tiles_per_district={MAX_TILES_PER_DISTRICT}')
print(f'[config] epochs={EPOCHS} batch_size_per_gpu={BATCH_SIZE_PER_GPU} global_batch_size={GLOBAL_BATCH_SIZE}')
print(f'[config] lr={LEARNING_RATE} lr_step_size={LR_STEP_SIZE} lr_gamma={LR_GAMMA}')
print(f'[config] val_split={VAL_SPLIT} test_split={TEST_SPLIT}')
print(f'[config] work_root={WORK_ROOT}')
print(f'[config] input_root={INPUT_ROOT}')
print(f'[gpu] count={CUDA_DEVICE_COUNT} selected={TRAIN_DEVICE} ddp_world_size={DDP_WORLD_SIZE}')
for device_index in range(CUDA_DEVICE_COUNT):
    print(f'[gpu] {device_index}: {torch.cuda.get_device_name(device_index)}')

## 3. Locate uploaded Kaggle Dataset files

Upload GADM, Overture, the six z18 district rasters, and preferably `building_footprints_usa_base.pth`. The notebook does not hardcode the Kaggle Dataset slug.

In [ ]:
def candidate_roots():
    roots = [INPUT_ROOT]
    local_geoai_data = Path.cwd() / 'geoai_data'
    if local_geoai_data.exists():
        roots.append(local_geoai_data)
    return roots


def find_input_file(filename, required=True):
    matches = []
    for root in candidate_roots():
        if root.exists():
            matches.extend(path for path in root.rglob(filename) if path.is_file())
    if matches:
        selected = sorted(matches, key=lambda path: len(str(path)))[0]
        print(f'[input] {filename}: {selected}')
        return selected
    if required:
        searched = ', '.join(str(root) for root in candidate_roots())
        raise FileNotFoundError(f'Missing required file {filename}. Searched: {searched}')
    print(f'[input] optional file not found: {filename}')
    return None


log_step(3, 'Locate uploaded Kaggle Dataset files')

DISTRICTS_PATH = find_input_file('gadm41_danang_districts.geojson')
OVERTURE_GPKG_PATH = find_input_file('overture_danang.gpkg')
RASTER_PATHS = {
    district_id: find_input_file(f'{district_id}_z18.tif')
    for district_id in DISTRICT_IDS
}
BASE_MODEL_PATH = find_input_file('building_footprints_usa_base.pth', required=False)

for excluded_raster in EXCLUDED_RASTERS:
    excluded_path = find_input_file(excluded_raster, required=False)
    if excluded_path is not None:
        print(f'[input] ignoring excluded raster {excluded_raster}: {excluded_path}')

SMOKE_RASTER_PATH = RASTER_PATHS[SMOKE_DISTRICT_ID]
print(f'[input] smoke_raster={SMOKE_RASTER_PATH}')

## 4. Clip Overture building labels to z18 districts

This creates one single-class vector label file per district with `class=1`, then filters weak labels by source confidence, building area, and a small inward buffer.

In [ ]:
def normalize_text(value):
    text = unicodedata.normalize('NFKD', str(value))
    text = text.encode('ascii', 'ignore').decode('ascii').lower()
    return ''.join(char for char in text if char.isalnum())


def select_district(districts, district_id):
    target = normalize_text(district_id)
    columns = [
        column for column in [
            'admin_id', 'district_id', 'name',
            'NAME_3', 'VARNAME_3', 'NAME_2', 'VARNAME_2', 'NAME_1', 'VARNAME_1',
        ]
        if column in districts.columns
    ]
    for column in columns:
        mask = districts[column].map(normalize_text).eq(target)
        if mask.any():
            selected = districts.loc[mask].iloc[[0]].copy()
            print(f'[district] matched {district_id} by column {column}')
            return selected
    available = {column: sorted(districts[column].astype(str).unique())[:20] for column in columns}
    raise ValueError(f'Could not find district {district_id}. Available values: {available}')


def make_geometry_valid(geometry):
    if geometry is None or geometry.is_empty:
        return None
    if geometry.is_valid:
        return geometry
    try:
        from shapely.validation import make_valid
        return make_valid(geometry)
    except Exception:
        return geometry.buffer(0)


def parse_source_records(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        return [value]
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        parsed = ast.literal_eval(value)
    except (SyntaxError, ValueError):
        return []
    if isinstance(parsed, list):
        return parsed
    if isinstance(parsed, dict):
        return [parsed]
    return []


def source_confidence(value):
    confidences = []
    for record in parse_source_records(value):
        if isinstance(record, dict) and record.get('confidence') is not None:
            try:
                confidences.append(float(record['confidence']))
            except (TypeError, ValueError):
                pass
    return max(confidences) if confidences else np.nan


def require_non_empty(gdf, stage, district_id):
    if gdf.empty:
        raise RuntimeError(f'No building labels remain for {district_id} after {stage}; lower the label filters and rerun')
    return gdf


def load_and_filter_district_labels(district_id, district_geometry):
    try:
        buildings = gpd.read_file(OVERTURE_GPKG_PATH, layer='buildings', bbox=district_geometry.bounds)
    except Exception as exc:
        try:
            import pyogrio
            print('[overture] available layers:', pyogrio.list_layers(str(OVERTURE_GPKG_PATH)))
        except Exception:
            pass
        raise exc

    if buildings.crs is None:
        buildings = buildings.set_crs('EPSG:4326')
    buildings = buildings.to_crs('EPSG:4326')
    buildings = buildings[buildings.geometry.notna() & ~buildings.geometry.is_empty].copy()

    clipped = buildings[buildings.geometry.intersects(district_geometry)].copy()
    if clipped.empty:
        raise RuntimeError(f'No Overture building polygons intersect {district_id}')

    clipped['geometry'] = clipped.geometry.intersection(district_geometry)
    clipped['geometry'] = clipped.geometry.map(make_geometry_valid)
    clipped = clipped[clipped.geometry.notna() & ~clipped.geometry.is_empty].copy()
    clipped = clipped.explode(index_parts=False, ignore_index=True)
    clipped = clipped[clipped.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
    clipped['geometry'] = clipped.geometry.map(make_geometry_valid)
    clipped = clipped[clipped.geometry.notna() & ~clipped.geometry.is_empty].copy()
    clipped['class'] = 1

    label_filter_summary = {
        'district': district_id,
        'original': int(len(clipped)),
    }

    if 'sources' in clipped.columns:
        clipped['source_confidence'] = clipped['sources'].map(source_confidence)
        known_confidence = clipped['source_confidence'].notna().sum()
        if known_confidence > 0:
            clipped = clipped[
                clipped['source_confidence'].isna() |
                (clipped['source_confidence'] >= LABEL_CONFIDENCE_MIN)
            ].copy()
            require_non_empty(clipped, 'source confidence filter', district_id)
        label_filter_summary['known_confidence'] = int(known_confidence)
        label_filter_summary['after_confidence'] = int(len(clipped))
    else:
        label_filter_summary['known_confidence'] = 0
        label_filter_summary['after_confidence'] = int(len(clipped))
        print(f'[labels/{district_id}] no sources column; skipping confidence filter')

    clipped_utm = clipped.to_crs('EPSG:32648')
    clipped_utm['area_m2'] = clipped_utm.geometry.area
    clipped_utm = clipped_utm[
        clipped_utm['area_m2'].between(BUILDING_AREA_MIN_M2, BUILDING_AREA_MAX_M2)
    ].copy()
    require_non_empty(clipped_utm, 'building area filter', district_id)
    label_filter_summary['after_area'] = int(len(clipped_utm))

    if NEGATIVE_BUFFER_M > 0:
        clipped_utm['geometry'] = clipped_utm.geometry.buffer(-NEGATIVE_BUFFER_M)
        clipped_utm['geometry'] = clipped_utm.geometry.map(make_geometry_valid)
        clipped_utm = clipped_utm[
            clipped_utm.geometry.notna() &
            ~clipped_utm.geometry.is_empty &
            (clipped_utm.geometry.area > 4)
        ].copy()
        clipped_utm = clipped_utm[clipped_utm.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].copy()
        require_non_empty(clipped_utm, 'negative buffer filter', district_id)
    label_filter_summary['after_buffer'] = int(len(clipped_utm))

    filtered = clipped_utm.to_crs('EPSG:4326')
    filtered['class'] = 1
    return filtered[['class', 'geometry']], label_filter_summary


log_step(4, 'Clip and filter Overture weak labels for z18 districts')

if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

districts = gpd.read_file(DISTRICTS_PATH)
if districts.crs is None:
    districts = districts.set_crs('EPSG:4326')
districts = districts.to_crs('EPSG:4326')

DISTRICT_BOUNDARIES = {}
VECTOR_PATHS = {}
LABEL_FILTER_SUMMARIES = {}

for district_id in DISTRICT_IDS:
    district = select_district(districts, district_id)
    district_geometry = make_geometry_valid(district.geometry.iloc[0])
    DISTRICT_BOUNDARIES[district_id] = district_geometry

    filtered_labels, label_summary = load_and_filter_district_labels(district_id, district_geometry)
    vector_path = VECTOR_DIR / f'{district_id}_buildings.geojson'
    filtered_labels.to_file(vector_path, driver='GeoJSON')
    VECTOR_PATHS[district_id] = vector_path
    LABEL_FILTER_SUMMARIES[district_id] = label_summary

    print(f'[labels/{district_id}] filters={json.dumps(label_summary)}')
    print(f'[labels/{district_id}] final_buildings={len(filtered_labels)} vector={vector_path}')

## 5. Export GeoTIFF training tiles

Tiles are exported per district and copied into a combined `images/` and `labels/` dataset. The held-out test split is spatial inside each district.

In [ ]:
log_step(5, 'Export z18 GeoTIFF tiles and create spatial test holdout')


def reset_directory(directory):
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)


def tile_center_x(tile_path):
    with rasterio.open(tile_path) as src:
        bounds = src.bounds
    return (bounds.left + bounds.right) / 2


for split_dir in [RAW_TILES_ROOT, IMAGES_DIR, LABELS_DIR, TRAIN_IMAGES_DIR, TRAIN_LABELS_DIR, TEST_IMAGES_DIR, TEST_LABELS_DIR]:
    reset_directory(split_dir)

DISTRICT_TILE_SUMMARIES = {}

for district_id in DISTRICT_IDS:
    district_out = RAW_TILES_ROOT / district_id
    district_out.mkdir(parents=True, exist_ok=True)
    print(f'[tiles/{district_id}] raster={RASTER_PATHS[district_id]}')

    tiles = geoai.export_geotiff_tiles(
        in_raster=str(RASTER_PATHS[district_id]),
        out_folder=str(district_out),
        in_class_data=str(VECTOR_PATHS[district_id]),
        tile_size=TILE_SIZE,
        stride=STRIDE,
        class_value_field='class',
        buffer_radius=0,
        max_tiles=MAX_TILES_PER_DISTRICT,
        metadata_format='PASCAL_VOC',
        skip_empty_tiles=True,
        quiet=False,
    )

    source_images_dir = district_out / 'images'
    source_labels_dir = district_out / 'labels'
    source_images = sorted(source_images_dir.glob('*.tif'))
    copied = 0
    for image_path in source_images:
        label_path = source_labels_dir / image_path.name
        if not label_path.exists():
            continue
        dest_name = f'{district_id}_{image_path.name}'
        shutil.copy2(image_path, IMAGES_DIR / dest_name)
        shutil.copy2(label_path, LABELS_DIR / dest_name)
        copied += 1

    DISTRICT_TILE_SUMMARIES[district_id] = {
        'export_return_count': None if tiles is None else int(len(tiles)),
        'copied_positive_tiles': copied,
        'max_tiles_per_district': MAX_TILES_PER_DISTRICT,
    }
    print(f'[tiles/{district_id}] copied_positive_tiles={copied}')


for split_dir in [TRAIN_IMAGES_DIR, TRAIN_LABELS_DIR, TEST_IMAGES_DIR, TEST_LABELS_DIR]:
    reset_directory(split_dir)

all_image_tiles = sorted(IMAGES_DIR.glob('*.tif'))
if len(all_image_tiles) < 8:
    raise RuntimeError(f'Need at least 8 positive tiles to create train/test splits; found {len(all_image_tiles)}')

tiles_by_district = defaultdict(list)
for image_path in all_image_tiles:
    district_id = next((item for item in DISTRICT_IDS if image_path.name.startswith(f'{item}_')), None)
    if district_id is None:
        raise RuntimeError(f'Could not infer district from tile name: {image_path.name}')
    tiles_by_district[district_id].append(image_path)

split_summary = {}
for district_id, district_tiles in tiles_by_district.items():
    ranked_tiles = sorted((tile_center_x(path), path) for path in district_tiles)
    test_count = max(1, int(math.ceil(len(ranked_tiles) * TEST_SPLIT)))
    if test_count >= len(ranked_tiles):
        test_count = max(1, len(ranked_tiles) - 1)
    test_names = {path.name for _, path in ranked_tiles[-test_count:]}

    train_count = 0
    test_count_actual = 0
    for _, image_path in ranked_tiles:
        label_path = LABELS_DIR / image_path.name
        if not label_path.exists():
            continue
        if image_path.name in test_names:
            shutil.copy2(image_path, TEST_IMAGES_DIR / image_path.name)
            shutil.copy2(label_path, TEST_LABELS_DIR / image_path.name)
            test_count_actual += 1
        else:
            shutil.copy2(image_path, TRAIN_IMAGES_DIR / image_path.name)
            shutil.copy2(label_path, TRAIN_LABELS_DIR / image_path.name)
            train_count += 1
    split_summary[district_id] = {
        'train_tiles': train_count,
        'test_tiles': test_count_actual,
        'test_split': TEST_SPLIT,
    }
    print(f'[split/{district_id}] train_tiles={train_count} test_tiles={test_count_actual}')

print(f'[tiles] images_dir={IMAGES_DIR}')
print(f'[tiles] labels_dir={LABELS_DIR}')
print(f'[split] train_images={TRAIN_IMAGES_DIR}')
print(f'[split] test_images={TEST_IMAGES_DIR}')

## 6. Validate train/test dataset before training

This catches missing labels, mismatched basenames, bad raster dimensions, and all-empty masks before a long GPU run.

In [ ]:
def tif_files(directory):
    return sorted(path for path in directory.glob('*.tif') if path.is_file())


def positive_pixel_count(label_path):
    with rasterio.open(label_path) as src:
        data = src.read(1)
    return int(np.count_nonzero(data))


def district_counts(paths):
    counts = {district_id: 0 for district_id in DISTRICT_IDS}
    for path in paths:
        district_id = next((item for item in DISTRICT_IDS if path.name.startswith(f'{item}_')), None)
        if district_id:
            counts[district_id] += 1
    return counts


log_step(6, 'Validate train and weak-label test tiles')


def validate_tile_pair_dir(images_dir, labels_dir, split_name):
    image_paths = tif_files(images_dir)
    label_paths = tif_files(labels_dir)
    image_stems = {path.stem for path in image_paths}
    label_stems = {path.stem for path in label_paths}

    if not image_paths:
        raise RuntimeError(f'No image tiles found in {images_dir}')
    if image_stems != label_stems:
        raise RuntimeError({
            'split': split_name,
            'missing_labels': sorted(image_stems - label_stems)[:20],
            'extra_labels': sorted(label_stems - image_stems)[:20],
        })

    positive_labels = 0
    total_positive_pixels = 0
    for index, image_path in enumerate(image_paths, start=1):
        label_path = labels_dir / image_path.name
        with rasterio.open(image_path) as image_src, rasterio.open(label_path) as label_src:
            if image_src.width != label_src.width or image_src.height != label_src.height:
                raise RuntimeError(f'Shape mismatch in {split_name}: {image_path.name}')
            if image_src.count < 3:
                raise RuntimeError(f'Image tile has fewer than 3 bands in {split_name}: {image_path.name}')
            if image_src.crs is None:
                raise RuntimeError(f'Image tile is missing CRS in {split_name}: {image_path.name}')
        positives = positive_pixel_count(label_path)
        total_positive_pixels += positives
        if positives > 0:
            positive_labels += 1
        if index % 500 == 0:
            print(f'[validate/{split_name}] checked {index}/{len(image_paths)} tiles')

    if positive_labels == 0:
        raise RuntimeError(f'All {split_name} label masks are empty')

    split_summary_local = {
        'image_tiles': len(image_paths),
        'label_tiles': len(label_paths),
        'positive_label_tiles': positive_labels,
        'total_positive_pixels': total_positive_pixels,
        'district_counts': district_counts(image_paths),
    }
    print(f'[validate/{split_name}] {json.dumps(split_summary_local)}')
    return split_summary_local


train_validation_summary = validate_tile_pair_dir(TRAIN_IMAGES_DIR, TRAIN_LABELS_DIR, 'train')
test_validation_summary = validate_tile_pair_dir(TEST_IMAGES_DIR, TEST_LABELS_DIR, 'test')

validation_summary = {
    'district_ids': DISTRICT_IDS,
    'excluded_rasters': EXCLUDED_RASTERS,
    'train': train_validation_summary,
    'test': test_validation_summary,
    'split_summary': split_summary,
    'test_split': TEST_SPLIT,
    'test_eval_max_tiles': TEST_EVAL_MAX_TILES,
    'tile_size': TILE_SIZE,
    'stride': STRIDE,
    'max_tiles_per_district': MAX_TILES_PER_DISTRICT,
    'district_tile_summaries': DISTRICT_TILE_SUMMARIES,
    'raster_paths': {district_id: str(path) for district_id, path in RASTER_PATHS.items()},
    'label_filters': {
        'confidence_min': LABEL_CONFIDENCE_MIN,
        'area_min_m2': BUILDING_AREA_MIN_M2,
        'area_max_m2': BUILDING_AREA_MAX_M2,
        'negative_buffer_m': NEGATIVE_BUFFER_M,
        'summary_by_district': LABEL_FILTER_SUMMARIES,
    },
}

summary_path = DATASET_ROOT / 'dataset_validation_summary.json'
with summary_path.open('w', encoding='utf-8') as fp:
    json.dump(validation_summary, fp, indent=2)

print(json.dumps(validation_summary, indent=2))
print(f'[validate] summary={summary_path}')

## 7. Fine-tune Mask R-CNN with DDP

This writes a small training script and launches `torch.distributed.run` when Kaggle exposes two GPUs. DDP splits the batch by process, so it avoids the `DataParallel` channel-splitting issue with torchvision detection models. Learning rate is dropped with `StepLR`.

In [ ]:
def ensure_base_model_weights():
    if BASE_MODEL_PATH is not None and BASE_MODEL_PATH.exists():
        print(f'[model] using uploaded base weights: {BASE_MODEL_PATH}')
        return BASE_MODEL_PATH

    target_path = RUN_ROOT / 'models' / 'building_footprints_usa_base.pth'
    if target_path.exists():
        return target_path

    print('[model] base weights were not uploaded; materializing via BuildingFootprintExtractor')
    print('[model] this may require Kaggle internet access the first time')
    from geoai.extract import BuildingFootprintExtractor
    target_path.parent.mkdir(parents=True, exist_ok=True)
    extractor = BuildingFootprintExtractor(device=str(TRAIN_DEVICE))
    torch.save(extractor.model.state_dict(), target_path)
    return target_path


def clean_model_dir():
    if MODEL_DIR.exists():
        shutil.rmtree(MODEL_DIR)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)


DDP_SCRIPT_SOURCE = r'''
import argparse
import datetime
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, DistributedSampler

from geoai.train import (
    ObjectDetectionDataset,
    collate_fn,
    evaluate,
    get_instance_segmentation_model,
    get_transform,
)


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--images-dir', required=True)
    parser.add_argument('--labels-dir', required=True)
    parser.add_argument('--output-dir', required=True)
    parser.add_argument('--pretrained-model-path', required=True)
    parser.add_argument('--epochs', type=int, required=True)
    parser.add_argument('--batch-size-per-gpu', type=int, required=True)
    parser.add_argument('--learning-rate', type=float, required=True)
    parser.add_argument('--lr-step-size', type=int, required=True)
    parser.add_argument('--lr-gamma', type=float, required=True)
    parser.add_argument('--val-split', type=float, required=True)
    parser.add_argument('--seed', type=int, required=True)
    parser.add_argument('--num-workers', type=int, required=True)
    parser.add_argument('--num-classes', type=int, default=2)
    parser.add_argument('--num-channels', type=int, default=3)
    return parser.parse_args()


def setup_distributed():
    distributed = 'RANK' in os.environ and 'WORLD_SIZE' in os.environ
    if not distributed:
        return False, 0, 0, 1

    rank = int(os.environ['RANK'])
    local_rank = int(os.environ.get('LOCAL_RANK', 0))
    world_size = int(os.environ['WORLD_SIZE'])
    backend = 'nccl' if torch.cuda.is_available() else 'gloo'
    dist.init_process_group(backend=backend, timeout=datetime.timedelta(minutes=180))
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
    return True, rank, local_rank, world_size


def cleanup_distributed(distributed):
    if distributed and dist.is_initialized():
        dist.destroy_process_group()


def is_rank0(rank):
    return rank == 0


def normalized_state_dict(checkpoint):
    state_dict = checkpoint.get('model_state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
    if not isinstance(state_dict, dict):
        raise TypeError('Unsupported checkpoint format')
    if any(key.startswith('module.') for key in state_dict.keys()):
        return {key.removeprefix('module.'): value for key, value in state_dict.items()}
    return state_dict


def load_pretrained_weights(model, model_path, device, rank):
    checkpoint = torch.load(model_path, map_location=device)
    state_dict = normalized_state_dict(checkpoint)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if is_rank0(rank):
        print(f'[model] loaded pretrained weights from {model_path}', flush=True)
        print(f'[model] missing_keys={len(missing)} unexpected_keys={len(unexpected)}', flush=True)


def paired_tile_paths(images_dir, labels_dir):
    image_paths = sorted(Path(images_dir).glob('*.tif'))
    label_by_name = {path.name: path for path in Path(labels_dir).glob('*.tif')}
    missing = [path.name for path in image_paths if path.name not in label_by_name]
    if missing:
        raise RuntimeError(f'Missing label files for {len(missing)} images; examples={missing[:10]}')
    label_paths = [label_by_name[path.name] for path in image_paths]
    if not image_paths:
        raise RuntimeError(f'No training images found in {images_dir}')
    return [str(path) for path in image_paths], [str(path) for path in label_paths]


def split_train_val(image_paths, label_paths, val_split, seed):
    indices = list(range(len(image_paths)))
    rng = random.Random(seed)
    rng.shuffle(indices)
    val_count = max(1, int(math.ceil(len(indices) * val_split))) if len(indices) > 1 else 0
    if val_count >= len(indices):
        val_count = max(1, len(indices) - 1)
    val_indices = set(indices[:val_count])
    train_images, train_labels, val_images, val_labels = [], [], [], []
    for idx, (image_path, label_path) in enumerate(zip(image_paths, label_paths)):
        if idx in val_indices:
            val_images.append(image_path)
            val_labels.append(label_path)
        else:
            train_images.append(image_path)
            train_labels.append(label_path)
    return train_images, train_labels, val_images, val_labels


def make_loader(dataset, batch_size, num_workers, sampler=None, shuffle=False):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        shuffle=shuffle if sampler is None else False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_fn,
    )


def reduce_loss(total_loss, batch_count, device, distributed):
    stats = torch.tensor([float(total_loss), float(batch_count)], device=device)
    if distributed:
        dist.all_reduce(stats, op=dist.ReduceOp.SUM)
    total = stats[0].item()
    count = max(1.0, stats[1].item())
    return total / count


def save_training_summary(output_dir, history, config, best_iou, final_metrics):
    summary_txt = Path(output_dir) / 'training_summary.txt'
    with summary_txt.open('w', encoding='utf-8') as fp:
        fp.write('Da Nang urban z18 Mask R-CNN DDP fine-tune\n')
        fp.write(f'epochs: {config["epochs"]}\n')
        fp.write(f'batch_size_per_gpu: {config["batch_size_per_gpu"]}\n')
        fp.write(f'learning_rate: {config["learning_rate"]}\n')
        fp.write(f'lr_step_size: {config["lr_step_size"]}\n')
        fp.write(f'lr_gamma: {config["lr_gamma"]}\n')
        fp.write(f'best_val_iou: {best_iou}\n')
        fp.write(f'final_metrics: {json.dumps(final_metrics)}\n')

    with (Path(output_dir) / 'training_history.json').open('w', encoding='utf-8') as fp:
        json.dump(history, fp, indent=2)


def main():
    args = parse_args()
    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    distributed, rank, local_rank, world_size = setup_distributed()
    device = torch.device(f'cuda:{local_rank}' if torch.cuda.is_available() else 'cpu')

    if is_rank0(rank):
        print(f'[ddp] distributed={distributed} world_size={world_size} device={device}', flush=True)
        print(f'[ddp] lr schedule StepLR(step_size={args.lr_step_size}, gamma={args.lr_gamma})', flush=True)

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    image_paths, label_paths = paired_tile_paths(args.images_dir, args.labels_dir)
    train_images, train_labels, val_images, val_labels = split_train_val(
        image_paths,
        label_paths,
        args.val_split,
        args.seed,
    )

    if is_rank0(rank):
        print(f'[data] train_tiles={len(train_images)} val_tiles={len(val_images)} total={len(image_paths)}', flush=True)

    train_dataset = ObjectDetectionDataset(
        train_images,
        train_labels,
        transforms=get_transform(train=True),
        num_channels=args.num_channels,
    )
    val_dataset = ObjectDetectionDataset(
        val_images,
        val_labels,
        transforms=get_transform(train=False),
        num_channels=args.num_channels,
    )

    train_sampler = DistributedSampler(
        train_dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        seed=args.seed,
    ) if distributed else None

    train_loader = make_loader(
        train_dataset,
        batch_size=args.batch_size_per_gpu,
        num_workers=args.num_workers,
        sampler=train_sampler,
        shuffle=not distributed,
    )
    val_loader = make_loader(
        val_dataset,
        batch_size=args.batch_size_per_gpu,
        num_workers=args.num_workers,
        sampler=None,
        shuffle=False,
    )

    model = get_instance_segmentation_model(
        num_classes=args.num_classes,
        num_channels=args.num_channels,
        pretrained=False,
    )
    load_pretrained_weights(model, args.pretrained_model_path, device, rank)
    model.to(device)

    ddp_model = DistributedDataParallel(
        model,
        device_ids=[local_rank] if torch.cuda.is_available() else None,
        output_device=local_rank if torch.cuda.is_available() else None,
        find_unused_parameters=True,
    ) if distributed else model

    params = [param for param in ddp_model.parameters() if param.requires_grad]
    optimizer = torch.optim.SGD(
        params,
        lr=args.learning_rate,
        momentum=0.9,
        weight_decay=0.0005,
    )
    scheduler = StepLR(optimizer, step_size=args.lr_step_size, gamma=args.lr_gamma)

    history = {
        'epochs': [],
        'train_loss': [],
        'val_loss': [],
        'val_iou': [],
        'lr': [],
        'world_size': world_size,
        'batch_size_per_gpu': args.batch_size_per_gpu,
        'global_batch_size': args.batch_size_per_gpu * world_size,
        'lr_step_size': args.lr_step_size,
        'lr_gamma': args.lr_gamma,
    }
    best_iou = -1.0
    started_at = time.time()

    for epoch in range(args.epochs):
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)

        ddp_model.train()
        local_total_loss = 0.0
        local_batch_count = 0
        step_total = len(train_loader)

        for step, (images, targets) in enumerate(train_loader, start=1):
            images = [image.to(device, non_blocking=True) for image in images]
            targets = [{key: value.to(device, non_blocking=True) for key, value in target.items()} for target in targets]

            loss_dict = ddp_model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            if not torch.isfinite(losses):
                if is_rank0(rank):
                    print(f'[train] non-finite loss at epoch={epoch + 1} step={step}; skipping batch', flush=True)
                continue

            optimizer.zero_grad(set_to_none=True)
            losses.backward()
            torch.nn.utils.clip_grad_norm_(params, max_norm=10.0)
            optimizer.step()

            local_total_loss += float(losses.item())
            local_batch_count += 1

            if is_rank0(rank) and (step == 1 or step == step_total or step % 25 == 0):
                print(
                    f'[train epoch {epoch + 1}/{args.epochs} step {step}/{step_total}] '
                    f'loss={losses.item():.4f} lr={optimizer.param_groups[0]["lr"]:.6g}',
                    flush=True,
                )

        train_loss = reduce_loss(local_total_loss, local_batch_count, device, distributed)

        final_metrics = {}
        if is_rank0(rank):
            eval_metrics = evaluate(model, val_loader, device, use_mask_iou=True) if len(val_dataset) else {'loss': float('nan'), 'IoU': 0.0}
            val_loss = float(eval_metrics.get('loss', float('nan')))
            val_iou = float(eval_metrics.get('IoU', 0.0))
            current_lr = float(optimizer.param_groups[0]['lr'])
            history['epochs'].append(epoch + 1)
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['val_iou'].append(val_iou)
            history['lr'].append(current_lr)

            print(
                f'[epoch {epoch + 1}/{args.epochs}] train_loss={train_loss:.4f} '
                f'val_loss={val_loss:.4f} val_iou={val_iou:.4f} lr={current_lr:.6g}',
                flush=True,
            )

            if val_iou > best_iou:
                best_iou = val_iou
                torch.save(model.state_dict(), output_dir / 'best_model.pth')
                print(f'[checkpoint] saved best_model.pth val_iou={best_iou:.4f}', flush=True)

            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'history': history,
            }, output_dir / 'last_checkpoint.pth')

        if distributed:
            dist.barrier()

        scheduler.step()

    if is_rank0(rank):
        torch.save(model.state_dict(), output_dir / 'final_model.pth')
        torch.save(history, output_dir / 'training_history.pth')
        if not (output_dir / 'best_model.pth').exists():
            torch.save(model.state_dict(), output_dir / 'best_model.pth')

        try:
            model.load_state_dict(torch.load(output_dir / 'best_model.pth', map_location=device))
            final_metrics = evaluate(model, val_loader, device, use_mask_iou=True) if len(val_dataset) else {'loss': float('nan'), 'IoU': 0.0}
        except Exception as exc:
            final_metrics = {'error': f'{type(exc).__name__}: {exc}'}

        history['elapsed_seconds'] = round(time.time() - started_at, 2)
        save_training_summary(
            output_dir,
            history,
            {
                'epochs': args.epochs,
                'batch_size_per_gpu': args.batch_size_per_gpu,
                'learning_rate': args.learning_rate,
                'lr_step_size': args.lr_step_size,
                'lr_gamma': args.lr_gamma,
            },
            best_iou,
            final_metrics,
        )
        print(f'[done] best_model={output_dir / "best_model.pth"}', flush=True)
        print(f'[done] final_model={output_dir / "final_model.pth"}', flush=True)
        print(f'[done] elapsed_seconds={history["elapsed_seconds"]}', flush=True)

    cleanup_distributed(distributed)


if __name__ == '__main__':
    main()
'''


def write_ddp_script():
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    DDP_TRAIN_SCRIPT.write_text(DDP_SCRIPT_SOURCE, encoding='utf-8')
    print(f'[train] wrote DDP script: {DDP_TRAIN_SCRIPT}')


def run_training_subprocess(base_model_path, distributed):
    if distributed:
        nproc = DDP_WORLD_SIZE
        cmd = [
            sys.executable,
            '-m',
            'torch.distributed.run',
            '--standalone',
            f'--nproc_per_node={nproc}',
            str(DDP_TRAIN_SCRIPT),
        ]
    else:
        nproc = 1
        cmd = [sys.executable, str(DDP_TRAIN_SCRIPT)]

    cmd.extend([
        '--images-dir', str(TRAIN_IMAGES_DIR),
        '--labels-dir', str(TRAIN_LABELS_DIR),
        '--output-dir', str(MODEL_DIR),
        '--pretrained-model-path', str(base_model_path),
        '--epochs', str(EPOCHS),
        '--batch-size-per-gpu', str(BATCH_SIZE_PER_GPU),
        '--learning-rate', str(LEARNING_RATE),
        '--lr-step-size', str(LR_STEP_SIZE),
        '--lr-gamma', str(LR_GAMMA),
        '--val-split', str(VAL_SPLIT),
        '--seed', str(SEED),
        '--num-workers', str(NUM_WORKERS_PER_GPU),
        '--num-classes', '2',
        '--num-channels', '3',
    ])

    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    print(f'[train] command={" ".join(cmd)}')
    print(f'[train] nproc={nproc} distributed={distributed}')
    subprocess.run(cmd, check=True, env=env)


base_model_path = ensure_base_model_weights()
training_mode = 'single_process'
log_step(7, 'Fine-tune Mask R-CNN with DDP and StepLR')
clean_model_dir()
write_ddp_script()

train_tile_count = train_validation_summary['image_tiles']
approx_train_tiles = math.ceil(train_tile_count * (1 - VAL_SPLIT))
approx_train_batches_per_gpu = math.ceil(approx_train_tiles / max(1, GLOBAL_BATCH_SIZE))
approx_total_optimizer_steps = EPOCHS * approx_train_batches_per_gpu
print(f'[train] train_tiles={train_tile_count} val_split={VAL_SPLIT} approx_train_tiles={approx_train_tiles}')
print(f'[train] batch_size_per_gpu={BATCH_SIZE_PER_GPU} world_size={DDP_WORLD_SIZE} global_batch_size={GLOBAL_BATCH_SIZE}')
print(f'[train] approx_batches_per_epoch={approx_train_batches_per_gpu} approx_total_optimizer_steps={approx_total_optimizer_steps}')
print(f'[train] scheduler=StepLR(step_size={LR_STEP_SIZE}, gamma={LR_GAMMA})')

use_distributed = USE_DDP and DDP_WORLD_SIZE >= 2
if use_distributed:
    try:
        print('[train] launching torch.distributed.run DDP; this uses both T4 GPUs without DataParallel channel splitting')
        run_training_subprocess(base_model_path, distributed=True)
        training_mode = f'ddp_{DDP_WORLD_SIZE}gpu'
    except subprocess.CalledProcessError as exc:
        print(f'[train] DDP failed with exit_code={exc.returncode}; falling back to single-process cuda:0/CPU')
        run_training_subprocess(base_model_path, distributed=False)
        training_mode = 'single_process_fallback'
else:
    print('[train] DDP disabled or fewer than two CUDA devices; training single-process on cuda:0/CPU')
    run_training_subprocess(base_model_path, distributed=False)

BEST_MODEL = MODEL_DIR / 'best_model.pth'
FINAL_MODEL = MODEL_DIR / 'final_model.pth'
HISTORY_JSON = MODEL_DIR / 'training_history.json'
if not BEST_MODEL.exists():
    raise FileNotFoundError(f'Missing best model checkpoint: {BEST_MODEL}')

print(f'[done] training_mode={training_mode}')
print(f'[done] best_model={BEST_MODEL}')
print(f'[done] final_model={FINAL_MODEL if FINAL_MODEL.exists() else "missing"}')
print(f'[done] history_json={HISTORY_JSON if HISTORY_JSON.exists() else "missing"}')

## 8. Test metrics and smoke test the fine-tuned model

This evaluates the held-out weak-label test set, then crops the center of the configured smoke raster and writes a GeoJSON prediction file.

In [ ]:
log_step(8, 'Evaluate weak-label test set and smoke test')


def write_empty_feature_collection(path):
    payload = {'type': 'FeatureCollection', 'features': []}
    with path.open('w', encoding='utf-8') as fp:
        json.dump(payload, fp)


def write_center_crop(raster_path, crop_path, window_size):
    with rasterio.open(raster_path) as src:
        width = min(window_size, src.width)
        height = min(window_size, src.height)
        col_off = max((src.width - width) // 2, 0)
        row_off = max((src.height - height) // 2, 0)
        window = Window(col_off=col_off, row_off=row_off, width=width, height=height)
        data = src.read(window=window)
        transform = src.window_transform(window)
        meta = src.meta.copy()
        meta.update({'height': height, 'width': width, 'transform': transform})

    with rasterio.open(crop_path, 'w', **meta) as dst:
        dst.write(data)


SMOKE_DIR.mkdir(parents=True, exist_ok=True)
CROP_PATH = SMOKE_DIR / f'{SMOKE_DISTRICT_ID}_smoke_crop.tif'
PREDICTIONS_PATH = SMOKE_DIR / 'finetuned_predictions.geojson'
TEST_METRICS_PATH = SMOKE_DIR / 'weak_label_test_metrics.json'
write_center_crop(SMOKE_RASTER_PATH, CROP_PATH, SMOKE_WINDOW_SIZE)

from geoai.extract import BuildingFootprintExtractor

extractor = BuildingFootprintExtractor(model_path=str(BEST_MODEL), device=str(TRAIN_DEVICE))


def safe_divide(numerator, denominator):
    return float(numerator / denominator) if denominator else 0.0


def prediction_mask_for_tile(image_path):
    with rasterio.open(image_path) as src:
        out_shape = (src.height, src.width)
        transform = src.transform
        crs = src.crs

    predicted = extractor.process_raster(
        str(image_path),
        batch_size=1,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        filter_edges=False,
    )
    detection_count = 0 if predicted is None else len(predicted)
    if predicted is None or predicted.empty:
        return np.zeros(out_shape, dtype=bool), detection_count

    if predicted.crs is None and crs is not None:
        predicted = predicted.set_crs(crs)
    elif crs is not None:
        predicted = predicted.to_crs(crs)

    shapes = [(geometry, 1) for geometry in predicted.geometry if geometry is not None and not geometry.is_empty]
    if not shapes:
        return np.zeros(out_shape, dtype=bool), detection_count

    mask = rasterize(
        shapes,
        out_shape=out_shape,
        transform=transform,
        fill=0,
        dtype='uint8',
    )
    return mask.astype(bool), detection_count


def evaluate_against_weak_labels():
    test_tiles = sorted(TEST_IMAGES_DIR.glob('*.tif'))
    eval_tiles = test_tiles[: min(TEST_EVAL_MAX_TILES, len(test_tiles))]
    if not eval_tiles:
        raise RuntimeError(f'No test tiles available in {TEST_IMAGES_DIR}')

    print(f'[test] evaluating {len(eval_tiles)}/{len(test_tiles)} weak-label test tiles')
    totals = {'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0, 'detections': 0}
    started_at = time.time()

    for index, image_path in enumerate(eval_tiles, start=1):
        label_path = TEST_LABELS_DIR / image_path.name
        with rasterio.open(label_path) as label_src:
            truth_mask = label_src.read(1) > 0

        pred_mask, detection_count = prediction_mask_for_tile(image_path)
        totals['tp'] += int(np.logical_and(pred_mask, truth_mask).sum())
        totals['fp'] += int(np.logical_and(pred_mask, ~truth_mask).sum())
        totals['fn'] += int(np.logical_and(~pred_mask, truth_mask).sum())
        totals['tn'] += int(np.logical_and(~pred_mask, ~truth_mask).sum())
        totals['detections'] += int(detection_count)
        print(f'[test {index}/{len(eval_tiles)}] tile={image_path.name} detections={detection_count}')

    tp, fp, fn, tn = totals['tp'], totals['fp'], totals['fn'], totals['tn']
    metrics = {
        'note': 'Metrics are against Overture weak labels, not hand-labeled ground truth.',
        'evaluated_tiles': len(eval_tiles),
        'available_test_tiles': len(test_tiles),
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'tp_pixels': tp,
        'fp_pixels': fp,
        'fn_pixels': fn,
        'tn_pixels': tn,
        'pixel_accuracy': safe_divide(tp + tn, tp + fp + fn + tn),
        'building_precision': safe_divide(tp, tp + fp),
        'building_recall': safe_divide(tp, tp + fn),
        'building_f1': safe_divide(2 * tp, 2 * tp + fp + fn),
        'building_iou': safe_divide(tp, tp + fp + fn),
        'avg_detections_per_tile': safe_divide(totals['detections'], len(eval_tiles)),
        'elapsed_seconds': round(time.time() - started_at, 2),
    }
    with TEST_METRICS_PATH.open('w', encoding='utf-8') as fp:
        json.dump(metrics, fp, indent=2)

    print('[test] weak-label metrics:')
    for key in ['pixel_accuracy', 'building_precision', 'building_recall', 'building_f1', 'building_iou', 'avg_detections_per_tile', 'elapsed_seconds']:
        print(f'[metric] {key}={metrics[key]}')
    print(f'[test] metrics_json={TEST_METRICS_PATH}')
    return metrics


test_metrics = evaluate_against_weak_labels()

predictions = extractor.process_raster(
    str(CROP_PATH),
    batch_size=1,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    filter_edges=False,
)

prediction_count = 0 if predictions is None else len(predictions)
if predictions is not None and not predictions.empty:
    if predictions.crs is None:
        predictions = predictions.set_crs('EPSG:4326')
    else:
        predictions = predictions.to_crs('EPSG:4326')
    predictions.to_file(PREDICTIONS_PATH, driver='GeoJSON')
else:
    write_empty_feature_collection(PREDICTIONS_PATH)

print(f'[smoke] detections={prediction_count}')
print(f'[smoke] crop={CROP_PATH}')
print(f'[smoke] predictions={PREDICTIONS_PATH}')

## 9. Package outputs

Download the zip from Kaggle output. Use `best_model.pth` in the app by setting `GEOAI_MODEL_PATH`.

In [ ]:
log_step(9, 'Package model artifacts')


def zip_directory(source_dir, zip_path):
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(source_dir.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(source_dir.parent))


training_summary = {
    'district_ids': DISTRICT_IDS,
    'excluded_rasters': EXCLUDED_RASTERS,
    'training_mode': training_mode,
    'ddp_world_size': DDP_WORLD_SIZE,
    'batch_size_per_gpu': BATCH_SIZE_PER_GPU,
    'global_batch_size': GLOBAL_BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'lr_step_size': LR_STEP_SIZE,
    'lr_gamma': LR_GAMMA,
    'best_model': str(BEST_MODEL),
    'final_model': str(FINAL_MODEL),
    'training_history_json': str(HISTORY_JSON),
    'ddp_train_script': str(DDP_TRAIN_SCRIPT),
    'smoke_predictions': str(PREDICTIONS_PATH),
    'weak_label_test_metrics': test_metrics,
    'weak_label_test_metrics_path': str(TEST_METRICS_PATH),
    'dataset_summary': validation_summary,
}
with (MODEL_DIR / 'kaggle_run_summary.json').open('w', encoding='utf-8') as fp:
    json.dump(training_summary, fp, indent=2)

zip_directory(MODEL_DIR, ZIP_PATH)

print(f'[artifact] zip={ZIP_PATH}')
print(f'[backend] set GEOAI_MODEL_PATH={BEST_MODEL}')